
Project : Pentaho Log Intelligence

Layer   : Silver

Notebook: 04_Silver_Transformation_localhostAccess

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Silver Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de Parametros 

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

BRONZE_TABLE_LOCALHOST = "pentaho_logs.bronze.bronze_localhostaccess"

SILVER_TABLE_LOCALHOST= "pentaho_logs.silver.silver_localhostaccess"

#### Lectura  de tabla Bronze 

In [0]:
df_localhost_sl = spark.table(BRONZE_TABLE_LOCALHOST).filter(col("file_name").isin(archivos_nuevos))

#display(df_localhost_sl.limit(20))

In [0]:
df_localhost_sl.printSchema()

#### Enriquecimiento Data Frame

In [0]:
from pyspark.sql.functions import (col,regexp_extract,to_timestamp,to_date,date_format,try_to_date)

df_silver_localhost = (df_localhost_sl.withColumn("usuario",regexp_extract(col("descripcion"), r"Username=([^,\]]+)",1))
                      .withColumn("Fecha_evento", date_format(try_to_date(regexp_extract(col("descripcion"),r"\[(\d{2}/[A-Za-z]{3}/\d{4})",1),"dd/MMM/yyyy"),"yyyy-MM-dd"))
                    
                       .withColumn("hora",regexp_extract(col("descripcion"),r"\[\d{2}/[A-Za-z]{3}/\d{4}:(\d{2}:\d{2}:\d{2})",1))
                     
                   )
#display(display(df_silver_localhost))
#se modifico to_date por try_to_date (validar)


#### validación DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_silver_localhost.count():,}")

# Estructura
df_silver_localhost.printSchema()

In [0]:
from pyspark.sql.functions import col, sum, when

df_silver_localhost.select(
    sum(when(col("usuario").isNull(), 1).otherwise(0)).alias("usuario_null")
).show()


#### Creación Tabla Silver localhost_Access

In [0]:
SILVER_TABLE_LOCALHOST = "pentaho_logs.silver.silver_localhostaccess"
(
    df_silver_localhost.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE_LOCALHOST)
)

In [0]:
#display(spark.table(SILVER_TABLE_LOCALHOST).limit(20))